In [ ]:
!pip install nltk
!pip install --upgrade sympy
!pip install --upgrade transformers
!pip install stanza
import nltk
from nltk.stem import WordNetLemmatizer#Lemmatization
from nltk.tokenize import word_tokenize #Lemmatization
import string
from sklearn.feature_extraction.text import TfidfVectorizer#TF-IDF
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re  #regular expression
import stanza
from concurrent.futures import ThreadPoolExecutor
import ast
from nltk.corpus.reader.wordnet import NOUN, VERB, ADJ, ADV

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.1
    Uninstalling sympy-1.13.1:
      Successfully uninstalled sympy-1.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires sympy==1.13.1; python_version >= "3.9", but you have sympy 1.14.0 which is incompatible.
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0


In [ ]:
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
def other_cards(soup):
  Name = soup.find("p", class_="card-text-title").text.strip()
  type_card_element = soup.find("p", class_="card-text-type")
  effect= soup.find("div", class_="card-text")
  #Extract text from each element in all_examples
  all_examples_text = [example.text for example in effect]
  #Split the extracted text into lines
  lines = []
  for text in all_examples_text:
      lines.extend(text.splitlines())
  lines = [line for line in lines if line.strip() != '']
  Effect = lines[3] if len(lines) > 3 else None

  #Check if the element exists before accessing its text
  if type_card_element:
      type_card = type_card_element.text.strip()
      type_card = type_card.split("-")
      Type = type_card[0].strip()
  HP=None
  Name_Attack=None
  Attack=None
  Energy1=None
  Energy2=None
  Energy3=None
  Energy4=None
  Weakness=None
  Resistance=None
  Retreat=None
  return Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance, Retreat

In [ ]:
def divide_line(line):
  #Divide the texts in separate lines
  lines = line.split(" ")
  #Remove empty rows
  lines = [lines.strip() for lines in lines if lines.strip()]
  return lines[1]

In [ ]:
def data_update(url):

  #Request to the page
  response = requests.get(url)

  if response.status_code == 200:
      #Analysis of page content
      soup = BeautifulSoup(response.text, 'html.parser')

      type_card_element = soup.find("p", class_="card-text-type")

      #Check if the element exists before accessing its text
      if type_card_element:
          type_card = type_card_element.text.strip()
          type_card = type_card.split("-")
          card_type = type_card[0].strip()
      if card_type == "Pokémon":

        #Informations extraction
        Name = soup.find("span", class_="card-text-name").text.strip()

        card_info_text = soup.find("p", class_="card-text-title").text.strip()

        card_info_parts = card_info_text.split("-")
        Type = card_info_parts[1].strip()
        HP = card_info_parts[2].strip()
        Energy=" "
        Attack=" "

        all_examples = soup.find_all("p", class_="card-text-attack-info")

        all_examples_text = [example.text for example in all_examples]

        lines = []
        for text in all_examples_text:
            lines.extend(text.splitlines())

        lines = [line.strip() for line in lines if line.strip()]

        lines_with_numbers = [line for line in lines if re.search(r'\d', line)]

        span_element = soup.find("span", class_="ptcg-symbol").text.strip()

        if lines_with_numbers and span_element!="0":

            if len(lines) >= 3:
              if len(lines_with_numbers)>=2:
                Energy = lines[2]
                ln=lines_with_numbers[1]

                match = re.search(r'^(.*?)(\d+)([^\d]*)$', ln.strip())
                if match:
                    Name_Attack = match.group(1).strip()
                    Attack = match.group(2).strip()
                    Special_Char = match.group(3).strip()

              else:
                Energy = lines[0]
                ln=lines_with_numbers[0]

                match = re.search(r'^(.*?)(\d+)([^\d]*)$', ln.strip())
                if match:
                    Name_Attack = match.group(1).strip()
                    Attack = match.group(2).strip()
                    Special_Char = match.group(3).strip()


            elif len(lines) == 2:
                Energy = lines[0]
                line_attack = lines[1]
                match = re.search(r'^(.*?)(\d+)([^\d]*)$', line_attack.strip())

                if match:
                  Name_Attack = match.group(1).strip()
                  Attack = match.group(2).strip()

            Effect=None


        else:
            if len(lines) >= 3:
              Energy = lines[0]
              Name_Attack = lines[1]
              energy = lines[2]
              attack = lines[3]

            elif len(lines) == 2:
                Energy = lines[0]
                Name_Attack = lines[1]

            else:
                print("Less than two elements with the specified class found.")
            Effect=soup.find("p", class_="card-text-attack-effect").text.strip()
            Attack=None

        card_d = soup.find("p", class_="card-text-wrr").text.splitlines()

        lines2 = [card_d.strip() for card_d in card_d if card_d.strip()]

        if len(lines2) >= 3:
            l_Weakness = lines2[0]
            l_Resistance = lines2[1]
            l_Retreat = lines2[2]
        else:
            print("The text contains less than three useful lines.")

        Energy1 = Energy[0] if len(Energy) > 0 else None
        Energy2 = Energy[1] if len(Energy) > 1 else None
        Energy3 = Energy[2] if len(Energy) > 2 else None
        Energy4 = Energy[3] if len(Energy) > 3 else None

        Weakness=divide_line(l_Weakness)
        Resistance=divide_line(l_Resistance)
        Retreat=divide_line(l_Retreat)


      else:
        Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance, Retreat=other_cards(soup)

  else:
      print(f"ERROR REQUEST: {response.status_code}")

  return Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance, Retreat

In [ ]:

pokemon_data={
    "Trainer_Name":[],
    "Pokemon_Name":[],
    "Type":[],
    "HP":[],
    "Name_Attack":[],
    "Attack":[],
    "Effect":[],
    "Energy1":[],
    "Energy2":[],
    "Energy3":[],
    "Energy4":[],
    "Weakness":[],
    "Resistance":[],
    "Retreat":[],
    "N_cards":[]
}
pokemon=pd.DataFrame(pokemon_data)

In [ ]:
urltrainer = "https://limitlesstcg.com/tournaments/476"
#Make a request to the deck page
response = requests.get(urltrainer)

if response.status_code == 200:

    soup = BeautifulSoup(response.text, 'html.parser')
    table = soup.find("table", class_="data-table striped")

    if table:
        rows = table.find_all("tr")
        for row in rows:
            Trainer=row.get("data-name")
            link_tags = row.find_all("a")

            if len(link_tags) > 1:
                deck_link = link_tags[1]["href"]

                if "deck" in deck_link:
                    full_url = f"https://limitlesstcg.com{deck_link}"
                    deck_response = requests.get(full_url)

                    if deck_response.status_code == 200:
                        deck_soup = BeautifulSoup(deck_response.text, 'html.parser')
                        core_cards = deck_soup.find_all("div", class_="core-card")

                        for card in core_cards:
                            card_link = card.find("a")["href"]
                            full_card_url = f"https://limitlesstcg.com{card_link}"
                            Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance, Retreat=data_update(full_card_url)
                            quantity_element = card.find("span", class_="share")

                            if quantity_element:
                                quantity = quantity_element.text.strip()
                                N_cards = quantity[0].strip()

                            else:
                                N_cards = None
                                print("Quantity element not found.")

                            pokemon.loc[len(pokemon)]=[Trainer, Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance, Retreat, N_cards]

    else:
        print("Table not found on page.")
else:
    print(f"ERROR REQUEST: {response.status_code}")


POKEMON DATASET CREATED FOR DAVIDE MIGGIANO

In [ ]:

pokemon_davide={
    "Pokemon_Name":[],
    "Type":[],
    "HP":[],
    "Name_Attack":[],
    "Attack":[],
    "Effect":[],
    "Energy1":[],
    "Energy2":[],
    "Energy3":[],
    "Energy4":[],
    "Weakness":[],
    "Resistance":[],
    "Retreat":[],
    "N_cards": []
}
pokemondav=pd.DataFrame(pokemon_davide)
full_url = f"https://limitlesstcg.com/cards/N3"
#Make a request to the deck page
deck_responsee = requests.get(full_url)
if deck_responsee.status_code == 200:
    deck_soup = BeautifulSoup(deck_responsee.text, 'html.parser')

    core_cardss_container = deck_soup.find("div", class_="card-search-grid")
    core_cardss = core_cardss_container.find_all("a")
    print(len(core_cardss))
    for card in core_cardss:
        card_link = card["href"]
        print(card_link)
        full_card_urll = f"https://limitlesstcg.com{card_link}"

        Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance, Retreat=data_update(full_card_urll)
        #Card qyuantity extraction (how many cards are of that same type)
        quantity_element = card.find("span", class_="share")
        if quantity_element:
            quantity = quantity_element.text.strip()
            N_cards = quantity[0].strip()

        else:
            N_cards = None
            print("Quantity element not found.")
        pokemondav.loc[len(pokemondav)]=[Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance, Retreat, N_cards]


66
/cards/N3/1
Quantity element not found.
/cards/N3/2
Quantity element not found.
/cards/N3/3
Quantity element not found.
/cards/N3/4
Quantity element not found.
/cards/N3/5
Quantity element not found.
/cards/N3/6
Quantity element not found.
/cards/N3/7
Quantity element not found.
/cards/N3/8
Quantity element not found.
/cards/N3/9
Quantity element not found.
/cards/N3/10
Quantity element not found.
/cards/N3/11
Quantity element not found.
/cards/N3/12
Quantity element not found.
/cards/N3/13
Quantity element not found.
/cards/N3/14
Quantity element not found.
/cards/N3/15
Quantity element not found.
/cards/N3/16
Quantity element not found.
/cards/N3/17
Quantity element not found.
/cards/N3/18
Quantity element not found.
/cards/N3/19
Quantity element not found.
/cards/N3/20
Quantity element not found.
/cards/N3/21
Quantity element not found.
/cards/N3/22
Quantity element not found.
/cards/N3/23
Quantity element not found.
/cards/N3/24
Quantity element not found.
/cards/N3/25
Quantity 

END POKEMON DATASET CREATED FOR DAVIDE MIGGIANO

In [ ]:
pokemon

,Trainer_Name,Pokemon_Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance,Retreat,N_cards
0,Keito Arai,Ralts,Psychic,70 HP,Psyshot,30,None,P,C,None,None,Darkness,Fighting,1,3
1,Keito Arai,Kirlia,Psychic,90 HP,Psychic,60,None,P,P,C,None,Darkness,Fighting,1,2
2,Keito Arai,Gardevoir ex,Psychic,310 HP,Miracle Force,190,None,P,P,C,None,Darkness,Fighting,2,2
3,Keito Arai,Munkidori,Psychic,110 HP,Mind Bend,60,None,P,C,None,None,Darkness,Fighting,1,3
4,Keito Arai,Scream Tail,Psychic,90 HP,Slap,30,None,P,None,None,None,Darkness,Fighting,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1022,Raymond Siswara Wong,Nest Ball,Trainer,None,None,None,Search your deck for a Bas...,None,None,None,None,None,None,None,3
1023,Raymond Siswara Wong,Pal Pad,Trainer,None,None,None,Shuffle up to 2 Supporter ...,None,None,None,None,None,None,None,2
1024,Raymond Siswara Wong,Night Stretcher,Trainer,None,None,None,Put a Pokémon or a Basic E...,None,None,None,None,None,None,None,1
1025,Raymond Siswara Wong,Hisuian Heavy Ball,Trainer,None,None,None,Look at your face-down Pri...,None,None,None,None,None,None,None,1


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
pokemon.to_csv('cele_pokemon.csv', index=False)

In [ ]:
pokemon_prova=pd.read_csv("cele_pokemon.csv")

In [ ]:
def clean_effect(text, word):
    if word in text:
        index = text.find(word)  #Find the location of the word
        end_index = text.find('.', index)  #Find the dot after the word
        if end_index != -1:
            return text[end_index + 1:].strip()  #Return the text after the dot
    return text  #Otherwise return original text

#Clean cards effects
pokemon_prova['Effect']=pokemon_prova['Effect'].fillna('').apply(lambda eff: clean_effect(eff, 'Prize'))
pokemon_prova['Effect']=pokemon_prova['Effect'].fillna('').apply(lambda eff: clean_effect(eff, 'If'))
pokemon_prova['Effect']=pokemon_prova['Effect'].fillna('').apply(lambda eff: clean_effect(eff, 'if'))

In [ ]:
pokemon_prova

,Trainer_Name,Pokemon_Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance,Retreat,N_cards
0,Keito Arai,Ralts,Psychic,70 HP,Psyshot,30.0,,P,C,NaN,NaN,Darkness,Fighting,1.0,3
1,Keito Arai,Kirlia,Psychic,90 HP,Psychic,60.0,,P,P,C,NaN,Darkness,Fighting,1.0,2
2,Keito Arai,Gardevoir ex,Psychic,310 HP,Miracle Force,190.0,,P,P,C,NaN,Darkness,Fighting,2.0,2
3,Keito Arai,Munkidori,Psychic,110 HP,Mind Bend,60.0,,P,C,NaN,NaN,Darkness,Fighting,1.0,3
4,Keito Arai,Scream Tail,Psychic,90 HP,Slap,30.0,,P,NaN,NaN,NaN,Darkness,Fighting,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1022,Raymond Siswara Wong,Nest Ball,Trainer,NaN,NaN,NaN,Search your deck for a Bas...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
1023,Raymond Siswara Wong,Pal Pad,Trainer,NaN,NaN,NaN,Shuffle up to 2 Supporter ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
1024,Raymond Siswara Wong,Night Stretcher,Trainer,NaN,NaN,NaN,Put a Pokémon or a Basic E...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1025,Raymond Siswara Wong,Hisuian Heavy Ball,Trainer,NaN,NaN,NaN,") Then, shuffle your face-down Prize cards.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


In [ ]:
def penn_to_wordnet(pos_tag):
    if pos_tag.startswith('N'):
        return NOUN
    elif pos_tag.startswith('V'):
        return VERB
    elif pos_tag.startswith('J'):
        return ADJ
    elif pos_tag.startswith('R'):
        return ADV
    else:
        return NOUN

In [ ]:
def lemmatization(text, pos_tags):
    wnl = WordNetLemmatizer()
    if not isinstance(text, str):
        return "", []
    #tokenization and lemmatization
    tokens = word_tokenize(text.translate(str.maketrans("","",string.punctuation)))
    wn_map = {w.lower(): penn_to_wordnet(tag) for w, tag in pos_tags}
    lemmatized_words = [
        wnl.lemmatize(tok, pos=wn_map.get(tok.lower(), NOUN))
        for tok in tokens
    ]
    lemmatized_text = " ".join(lemmatized_words)
    #reconstruct POS on lemmas
    lemmatized_pos_tags = [
        (wnl.lemmatize(word, pos=penn_to_wordnet(tag)).lower(), tag)
        for word, tag in pos_tags
    ]
    return lemmatized_text, lemmatized_pos_tags


In [ ]:

#Upload english language template
nlp_stanza = stanza.Pipeline('en', use_gpu=True)

#POS tagging with Stanza
def get_stanza_pos_tags(text):
    doc = nlp_stanza(text)
    pos_tags = []
    for sentence in doc.sentences:  #Access to sentences
        for word in sentence.words:  #Access to words
            pos_tags.append((word.text, word.xpos))
    return pos_tags

def process_batch(effects_batch):
    return [get_stanza_pos_tags(effect) for effect in effects_batch]

effects = pokemon_prova['Effect'].astype(str).tolist()

#Use ThreadPoolExecutor for parallel processing
with ThreadPoolExecutor() as executor:
    batch_size = 10  #Number of effects per parallel batch
    #Divide list of effects
    batches = [effects[i:i + batch_size] for i in range(0, len(effects), batch_size)]

    #POS tagging for each batch
    results = list(executor.map(process_batch, batches))

#Flattened result to obtain a list with POS
flattened_results = [item for sublist in results for item in sublist]

pokemon_prova['POS_Tags_stanza'] = pd.Series(flattened_results, index=pokemon_prova.index)

print(pokemon_prova[['Effect', 'POS_Tags_stanza']])


INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined                  |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | combined_charlm           |
| sentiment    | sstplus_charlm            |
| ner          | ontonotes-ww-multi_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: constituency
INFO:stanza:Loading: depparse
INFO:stanza:Loading: sentiment
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


                                                 Effect  \
0                                                         
1                                                         
2                                                         
3                                                         
4                                                         
...                                                 ...   
1022                      Search your deck for a Bas...   
1023                      Shuffle up to 2 Supporter ...   
1024                      Put a Pokémon or a Basic E...   
1025        ) Then, shuffle your face-down Prize cards.   
1026                                                      

                                        POS_Tags_stanza  
0                                                    []  
1                                                    []  
2                                                    []  
3                                                    []  
4

In [ ]:
pokemon_prova['POS_Lemmatized'] = None
for idx, row in pokemon_prova.iterrows():
    if isinstance(row['Effect'], str):
        lem_text, lem_pos = lemmatization(row['Effect'], row['POS_Tags_stanza'])
        pokemon_prova.at[idx, 'Effect'] = lem_text
        pokemon_prova.at[idx, 'POS_Lemmatized'] = lem_pos

pokemon_prova



,Trainer_Name,Pokemon_Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance,Retreat,N_cards,POS_Tags_stanza,POS_Lemmatized
0,Keito Arai,Ralts,Psychic,70 HP,Psyshot,30.0,,P,C,NaN,NaN,Darkness,Fighting,1.0,3,[],[]
1,Keito Arai,Kirlia,Psychic,90 HP,Psychic,60.0,,P,P,C,NaN,Darkness,Fighting,1.0,2,[],[]
2,Keito Arai,Gardevoir ex,Psychic,310 HP,Miracle Force,190.0,,P,P,C,NaN,Darkness,Fighting,2.0,2,[],[]
3,Keito Arai,Munkidori,Psychic,110 HP,Mind Bend,60.0,,P,C,NaN,NaN,Darkness,Fighting,1.0,3,[],[]
4,Keito Arai,Scream Tail,Psychic,90 HP,Slap,30.0,,P,NaN,NaN,NaN,Darkness,Fighting,1.0,1,[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1022,Raymond Siswara Wong,Nest Ball,Trainer,NaN,NaN,NaN,Search your deck for a Basic Pokémon and put i...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,"[(Search, VB), (your, PRP$), (deck, NN), (for,...","[(search, VB), (your, PRP$), (deck, NN), (for,..."
1023,Raymond Siswara Wong,Pal Pad,Trainer,NaN,NaN,NaN,Shuffle up to 2 Supporter card from your disca...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,"[(Shuffle, VB), (up, IN), (to, IN), (2, CD), (...","[(shuffle, VB), (up, IN), (to, IN), (2, CD), (..."
1024,Raymond Siswara Wong,Night Stretcher,Trainer,NaN,NaN,NaN,Put a Pokémon or a Basic Energy card from your...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,"[(Put, VB), (a, DT), (Pokémon, NNP), (or, CC),...","[(put, VB), (a, DT), (pokémon, NNP), (or, CC),..."
1025,Raymond Siswara Wong,Hisuian Heavy Ball,Trainer,NaN,NaN,NaN,Then shuffle your facedown Prize card,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,"[(), -RRB-), (Then, RB), (,, ,), (shuffle, VB)...","[(), -RRB-), (then, RB), (,, ,), (shuffle, VB)..."


In [ ]:
#Calculation for “Num_Energy” considering
def calcola_num_energy(row):
    if row[["Energy1", "Energy2", "Energy3", "Energy4"]].isna().all():
        return np.nan
    if row["Energy1"] == '0':
        return 0
    return row[["Energy1", "Energy2", "Energy3", "Energy4"]].notna().sum()

pokemon_prova["Num_Energy"] = pokemon_prova.apply(calcola_num_energy, axis=1)


In [ ]:
pokemon_prova

,Trainer_Name,Pokemon_Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance,Retreat,N_cards,POS_Tags_stanza,POS_Lemmatized,Num_Energy
0,Keito Arai,Ralts,Psychic,70 HP,Psyshot,30.0,,P,C,NaN,NaN,Darkness,Fighting,1.0,3,[],[],2.0
1,Keito Arai,Kirlia,Psychic,90 HP,Psychic,60.0,,P,P,C,NaN,Darkness,Fighting,1.0,2,[],[],3.0
2,Keito Arai,Gardevoir ex,Psychic,310 HP,Miracle Force,190.0,,P,P,C,NaN,Darkness,Fighting,2.0,2,[],[],3.0
3,Keito Arai,Munkidori,Psychic,110 HP,Mind Bend,60.0,,P,C,NaN,NaN,Darkness,Fighting,1.0,3,[],[],2.0
4,Keito Arai,Scream Tail,Psychic,90 HP,Slap,30.0,,P,NaN,NaN,NaN,Darkness,Fighting,1.0,1,[],[],1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1022,Raymond Siswara Wong,Nest Ball,Trainer,NaN,NaN,NaN,Search your deck for a Basic Pokémon and put i...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,"[(Search, VB), (your, PRP$), (deck, NN), (for,...","[(search, VB), (your, PRP$), (deck, NN), (for,...",NaN
1023,Raymond Siswara Wong,Pal Pad,Trainer,NaN,NaN,NaN,Shuffle up to 2 Supporter card from your disca...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,"[(Shuffle, VB), (up, IN), (to, IN), (2, CD), (...","[(shuffle, VB), (up, IN), (to, IN), (2, CD), (...",NaN
1024,Raymond Siswara Wong,Night Stretcher,Trainer,NaN,NaN,NaN,Put a Pokémon or a Basic Energy card from your...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,"[(Put, VB), (a, DT), (Pokémon, NNP), (or, CC),...","[(put, VB), (a, DT), (pokémon, NNP), (or, CC),...",NaN
1025,Raymond Siswara Wong,Hisuian Heavy Ball,Trainer,NaN,NaN,NaN,Then shuffle your facedown Prize card,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,"[(), -RRB-), (Then, RB), (,, ,), (shuffle, VB)...","[(), -RRB-), (then, RB), (,, ,), (shuffle, VB)...",NaN


In [ ]:
pokemon_prova["HP"] = pokemon_prova["HP"].str.extract(r"(\d+)")
pokemon_prova["HP"] = pokemon_prova["HP"].apply(lambda x: int(x.split()[0]) if isinstance(x, str) else None)


dare peso alle parole

In [ ]:
pokemon_prova

,Trainer_Name,Pokemon_Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance,Retreat,N_cards,POS_Tags_stanza,POS_Lemmatized,Num_Energy
0,Keito Arai,Ralts,Psychic,70.0,Psyshot,30.0,,P,C,NaN,NaN,Darkness,Fighting,1.0,3,[],[],2.0
1,Keito Arai,Kirlia,Psychic,90.0,Psychic,60.0,,P,P,C,NaN,Darkness,Fighting,1.0,2,[],[],3.0
2,Keito Arai,Gardevoir ex,Psychic,310.0,Miracle Force,190.0,,P,P,C,NaN,Darkness,Fighting,2.0,2,[],[],3.0
3,Keito Arai,Munkidori,Psychic,110.0,Mind Bend,60.0,,P,C,NaN,NaN,Darkness,Fighting,1.0,3,[],[],2.0
4,Keito Arai,Scream Tail,Psychic,90.0,Slap,30.0,,P,NaN,NaN,NaN,Darkness,Fighting,1.0,1,[],[],1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1022,Raymond Siswara Wong,Nest Ball,Trainer,NaN,NaN,NaN,Search your deck for a Basic Pokémon and put i...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,"[(Search, VB), (your, PRP$), (deck, NN), (for,...","[(search, VB), (your, PRP$), (deck, NN), (for,...",NaN
1023,Raymond Siswara Wong,Pal Pad,Trainer,NaN,NaN,NaN,Shuffle up to 2 Supporter card from your disca...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,"[(Shuffle, VB), (up, IN), (to, IN), (2, CD), (...","[(shuffle, VB), (up, IN), (to, IN), (2, CD), (...",NaN
1024,Raymond Siswara Wong,Night Stretcher,Trainer,NaN,NaN,NaN,Put a Pokémon or a Basic Energy card from your...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,"[(Put, VB), (a, DT), (Pokémon, NNP), (or, CC),...","[(put, VB), (a, DT), (pokémon, NNP), (or, CC),...",NaN
1025,Raymond Siswara Wong,Hisuian Heavy Ball,Trainer,NaN,NaN,NaN,Then shuffle your facedown Prize card,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,"[(), -RRB-), (Then, RB), (,, ,), (shuffle, VB)...","[(), -RRB-), (then, RB), (,, ,), (shuffle, VB)...",NaN


In [ ]:
pokemon_prova

,Trainer_Name,Pokemon_Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance,Retreat,N_cards,POS_Tags_stanza,POS_Lemmatized,Num_Energy
0,Keito Arai,Ralts,Psychic,70.0,Psyshot,30.0,,P,C,NaN,NaN,Darkness,Fighting,1.0,3,[],[],2.0
1,Keito Arai,Kirlia,Psychic,90.0,Psychic,60.0,,P,P,C,NaN,Darkness,Fighting,1.0,2,[],[],3.0
2,Keito Arai,Gardevoir ex,Psychic,310.0,Miracle Force,190.0,,P,P,C,NaN,Darkness,Fighting,2.0,2,[],[],3.0
3,Keito Arai,Munkidori,Psychic,110.0,Mind Bend,60.0,,P,C,NaN,NaN,Darkness,Fighting,1.0,3,[],[],2.0
4,Keito Arai,Scream Tail,Psychic,90.0,Slap,30.0,,P,NaN,NaN,NaN,Darkness,Fighting,1.0,1,[],[],1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1022,Raymond Siswara Wong,Nest Ball,Trainer,NaN,NaN,NaN,Search your deck for a Basic Pokémon and put i...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,"[(Search, VB), (your, PRP$), (deck, NN), (for,...","[(search, VB), (your, PRP$), (deck, NN), (for,...",NaN
1023,Raymond Siswara Wong,Pal Pad,Trainer,NaN,NaN,NaN,Shuffle up to 2 Supporter card from your disca...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,"[(Shuffle, VB), (up, IN), (to, IN), (2, CD), (...","[(shuffle, VB), (up, IN), (to, IN), (2, CD), (...",NaN
1024,Raymond Siswara Wong,Night Stretcher,Trainer,NaN,NaN,NaN,Put a Pokémon or a Basic Energy card from your...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,"[(Put, VB), (a, DT), (Pokémon, NNP), (or, CC),...","[(put, VB), (a, DT), (pokémon, NNP), (or, CC),...",NaN
1025,Raymond Siswara Wong,Hisuian Heavy Ball,Trainer,NaN,NaN,NaN,Then shuffle your facedown Prize card,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,"[(), -RRB-), (Then, RB), (,, ,), (shuffle, VB)...","[(), -RRB-), (then, RB), (,, ,), (shuffle, VB)...",NaN


In [ ]:
def fix_joined_words(text):
    text = re.sub(r'(\w)([A-Z])', r'\1 \2', text)  #Added a space between joined words
    return text
pokemon_prova['Effect']=pokemon_prova['Effect'].apply(fix_joined_words)

In [ ]:
pokemon_with_effect = pokemon_prova[pokemon_prova["Effect"].notna()]

#Effects for each trainer
trainer_effects = pokemon_with_effect.groupby("Trainer_Name")["Effect"].apply(lambda x: " ".join(x)).reset_index()

#Compute TF-IDF
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(trainer_effects["Effect"])

tfidf_df = pd.DataFrame(X.toarray(), index=trainer_effects["Trainer_Name"], columns=vectorizer.get_feature_names_out())

#Obtain 10 key words
def get_keywords(trainer):
    if trainer in tfidf_df.index:
        return tfidf_df.loc[trainer].nlargest(10).index.tolist()


def is_effect_present(effect):
    if pd.isna(effect):
        return False
    if isinstance(effect, str) and effect.strip() == "":
        return False
    if isinstance(effect, list) and len(effect) == 0:
        return False
    return True

pokemon_prova["KeyWord_Effect"] = pokemon_prova.apply(
    lambda row: get_keywords(row["Trainer_Name"]) if is_effect_present(row["Effect"]) else np.nan,
    axis=1
)

In [ ]:
pokemon_prova["HP"] = pd.to_numeric(pokemon_prova["HP"], errors="coerce")
pokemon_prova["Attack"] = pd.to_numeric(pokemon_prova["Attack"], errors="coerce")

In [ ]:
pokemon_prova.to_csv('pokemon_typecard.csv', index=False)


In [ ]:
pokemon_typecard=pd.read_csv("pokemon_typecard.csv")

In [ ]:
pokemon_typecard.to_csv('pokemon_typecard_post.csv', index=False)


In [ ]:
pokemon_typecard_post=pd.read_csv("pokemon_typecard_post.csv")

In [ ]:
pokemon_typecard_post

,Trainer_Name,Pokemon_Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance,Retreat,N_cards,POS_Tags_stanza,POS_Lemmatized,Num_Energy,KeyWord_Effect
0,Keito Arai,Ralts,Psychic,70.0,Psyshot,30.0,NaN,P,C,NaN,NaN,Darkness,Fighting,1.0,3,[],[],2.0,NaN
1,Keito Arai,Kirlia,Psychic,90.0,Psychic,60.0,NaN,P,P,C,NaN,Darkness,Fighting,1.0,2,[],[],3.0,NaN
2,Keito Arai,Gardevoir ex,Psychic,310.0,Miracle Force,190.0,NaN,P,P,C,NaN,Darkness,Fighting,2.0,2,[],[],3.0,NaN
3,Keito Arai,Munkidori,Psychic,110.0,Mind Bend,60.0,NaN,P,C,NaN,NaN,Darkness,Fighting,1.0,3,[],[],2.0,NaN
4,Keito Arai,Scream Tail,Psychic,90.0,Slap,30.0,NaN,P,NaN,NaN,NaN,Darkness,Fighting,1.0,1,[],[],1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1022,Raymond Siswara Wong,Nest Ball,Trainer,NaN,NaN,NaN,Search your deck for a Basic Pokémon and put i...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,"[('Search', 'VB'), ('your', 'PRP$'), ('deck', ...","[('search', 'VB'), ('your', 'PRP$'), ('deck', ...",NaN,"['card', 'deck', 'pokémon', 'number', 'opponen..."
1023,Raymond Siswara Wong,Pal Pad,Trainer,NaN,NaN,NaN,Shuffle up to 2 Supporter card from your disca...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,"[('Shuffle', 'VB'), ('up', 'IN'), ('to', 'IN')...","[('shuffle', 'VB'), ('up', 'IN'), ('to', 'IN')...",NaN,"['card', 'deck', 'pokémon', 'number', 'opponen..."
1024,Raymond Siswara Wong,Night Stretcher,Trainer,NaN,NaN,NaN,Put a Pokémon or a Basic Energy card from your...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,"[('Put', 'VB'), ('a', 'DT'), ('Pokémon', 'NNP'...","[('put', 'VB'), ('a', 'DT'), ('pokémon', 'NNP'...",NaN,"['card', 'deck', 'pokémon', 'number', 'opponen..."
1025,Raymond Siswara Wong,Hisuian Heavy Ball,Trainer,NaN,NaN,NaN,Then shuffle your facedown Prize card,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,"[(')', '-RRB-'), ('Then', 'RB'), (',', ','), (...","[(')', '-RRB-'), ('then', 'RB'), (',', ','), (...",NaN,"['card', 'deck', 'pokémon', 'number', 'opponen..."


In [ ]:
for index, value in pokemon_typecard['POS_Tags_stanza'].head(10).items():
    print('----')
    print(index)
    if isinstance(value, str):
      value = ast.literal_eval(value)
      for word, tag in value:
        if tag.startswith('VB'):
          print('VB', word)
        if tag.startswith('NN'):  #take VB, VBD, VBG, VBN, VBP, VBZ
          print('NN', word)
pokemon_typecard['POS_Tags_stanza'].head(10).items()

----
0
----
1
----
2
----
3
----
4
----
5
----
6
NN attack
VB does
NN damage
NN opponent
NN Pokémon
VB Do
VB apply
NN Weakness
NN Resistance
VB Benched
NN Pokémon
----
7
VB Discard
NN hand
VB draw
NN cards
----
8
----
9
VB Search
NN deck
NN Item
NN card
NN Pokémon
NN Tool
NN card
VB reveal
VB put
NN hand
VB shuffle
NN deck


In [ ]:
#Not null effects
effect_df = pokemon_typecard_post[pokemon_typecard_post['Effect'].notna()]
effect_df = effect_df.drop_duplicates(subset='Effect').copy()

effect_df['POS_Lemmatized'] = effect_df['POS_Lemmatized'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

sample = effect_df
print(sample)
#Creation of semantic frames
def smart_extract(effect, pos_tags):
    verb = None
    verb2 = None
    subject = None
    who=None
    sub=None
    object_ = None
    target_ = None
    number = None

    #Find verb
    for i, (word, tag) in enumerate(pos_tags):
        print('Effetto', effect)
        wordwho= re.compile(rf"opponents", re.IGNORECASE)
        match = wordwho.search(effect)
        if match:
            who = match.group()
            sub = 'player'
        else:
            who = 'player'


        if tag in ("MD",):
            #Find negation
            if i + 1 < len(pos_tags) and pos_tags[i + 1][0].lower() in ("not", "nt"):
                if i + 2 < len(pos_tags) and pos_tags[i + 2][1].startswith("VB"):
                    verb = f"cant {pos_tags[i + 2][0].lower()}"
                    break
            elif i + 1 < len(pos_tags) and pos_tags[i + 1][1].startswith("VB"):
                verb = f"{word.lower()} {pos_tags[i + 1][0].lower()}"
                break
        elif tag.startswith("VB") and not verb:
            verb = word.lower()

    #Find first subj and obj
    nouns = [word for word, tag in pos_tags if tag in ("NN", "NNS", "NNP", "NNPS", "PRP", "VBN")]
    if nouns:
      if sub:
        subject = f'{sub} {nouns[0]}'
      else:
        subject = nouns[0]
    if len(nouns) > 1 :
      object_ = nouns[1]
      if nouns[1] == "damage":
        object_ = nouns[2]



    #Find number
    match_number = re.search(r'\b(\d+)\b', effect)
    if match_number:
        number = match_number.group(1)
    print('obj', object_)

    if object_:
        pattern1 = re.compile(rf"{object_}\s+card", re.IGNORECASE)
        match = pattern1.search(effect)
        if match:
            object_ = match.group()
        pattern2 = re.compile(rf"{object_}\s+Pokémon", re.IGNORECASE)
        print(pattern1, pattern2)
        match = pattern2.search(effect)
        if match:
            object_ = match.group()

    if verb == "switch":
      named_nouns = [word for word, tag in pos_tags if tag in ("NN", "NNS", "NNP", "NNPS", "VBN")]
      if len(named_nouns) >= 5:
          object_ = f"{named_nouns[1]} {named_nouns[2]}-{named_nouns[3]} {named_nouns[4]}"

    any_match = re.search(r'\bany\s+(\w+\s+\w+|\w+)', effect, re.IGNORECASE)
    if any_match:
        object_ = any_match.group(1)

    if verb == "discard" and subject and "hand" in subject.lower():
      verb += "+draw"

    if verb == "discard+draw" and number and object_=="card":
        object_ = f"{object_}({number})"
    elif verb == "do"  and "damage" in effect and number:
        verb = f"do_damage{number}"

    #If the word deck is present then a column must be added
    #with a second action indicating that you must shuffle the deck after the first action
    if subject == 'deck':
      verb2 = 'shuffle_deck'


    return verb, subject, who ,object_, verb2


sample[['azione', 'chi', 'a_chi',  'cosa', 'azione2']] = sample.apply(
    lambda row: pd.Series(smart_extract(row['Effect'], row['POS_Lemmatized'])),
    axis=1
)

sample[['Effect', 'azione', 'chi','a_chi', 'cosa', 'azione2']]

sample[['Effect']].to_csv('sample.csv', index=False)


           Trainer_Name                    Pokemon_Name      Type     HP  \
6            Keito Arai                  Fezandipiti ex  Darkness  210.0   
7            Keito Arai            Professor's Research   Trainer    NaN   
9            Keito Arai                           Arven   Trainer    NaN   
10           Keito Arai                      Ultra Ball   Trainer    NaN   
11           Keito Arai                  Earthen Vessel   Trainer    NaN   
12           Keito Arai                       Nest Ball   Trainer    NaN   
13           Keito Arai                      Rare Candy   Trainer    NaN   
14           Keito Arai                 Night Stretcher   Trainer    NaN   
15           Keito Arai                       Super Rod   Trainer    NaN   
16           Keito Arai                 Counter Catcher   Trainer    NaN   
17           Keito Arai                      Secret Box   Trainer    NaN   
18           Keito Arai    Technical Machine: Evolution   Trainer    NaN   
19          

In [ ]:
#Merge between dataset
toooo = pokemon_typecard.merge(
    sample[['Effect', 'azione', 'chi', 'a_chi', 'cosa', 'azione2']],
    on='Effect',
    how='left'
)


In [ ]:
toooo.to_csv("Pokeomn_vero.csv", index=False)

In [ ]:
#Change type of file to JASON
toooo.to_json("Pokemon.json", orient="records", force_ascii=False, indent=4)

In [ ]:

toooo.head(22)

,Trainer_Name,Pokemon_Name,Type,HP,Name_Attack,Attack,Effect,Energy1,Energy2,Energy3,Energy4,Weakness,Resistance,Retreat,N_cards,POS_Tags_stanza,POS_Lemmatized,Num_Energy,KeyWord_Effect,azione,chi,a_chi,cosa,azione2
0,Keito Arai,Ralts,Psychic,70.0,Psyshot,30.0,NaN,P,C,NaN,NaN,Darkness,Fighting,1.0,3,[],[],2.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Keito Arai,Kirlia,Psychic,90.0,Psychic,60.0,NaN,P,P,C,NaN,Darkness,Fighting,1.0,2,[],[],3.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Keito Arai,Gardevoir ex,Psychic,310.0,Miracle Force,190.0,NaN,P,P,C,NaN,Darkness,Fighting,2.0,2,[],[],3.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Keito Arai,Munkidori,Psychic,110.0,Mind Bend,60.0,NaN,P,C,NaN,NaN,Darkness,Fighting,1.0,3,[],[],2.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Keito Arai,Scream Tail,Psychic,90.0,Slap,30.0,NaN,P,NaN,NaN,NaN,Darkness,Fighting,1.0,1,[],[],1.0,NaN,NaN,NaN,NaN,NaN,NaN
5,Keito Arai,Lillie's Clefairy ex,Psychic,190.0,Full Moon Rondo,20.0,NaN,P,C,NaN,NaN,Metal,none,1.0,1,[],[],2.0,NaN,NaN,NaN,NaN,NaN,NaN
6,Keito Arai,Fezandipiti ex,Darkness,210.0,Cruel Arrow,NaN,This attack do 100 damage to 1 of your opponent Pokémon Dont apply Weakness and Resistance for Benched Pokémon,C,C,C,NaN,Fighting,none,1.0,1,"[('This', 'DT'), ('attack', 'NN'), ('does', 'VBZ'), ('100', 'CD'), ('damage', 'NN'), ('to', 'IN'), ('1', 'CD'), ('of', 'IN'), ('your', 'PRP$'), ('opponent', 'NN'), (""'s"", 'POS'), ('Pokémon', 'NNP'), ('.', '.'), ('(', '-LRB-'), ('Do', 'VB'), (""n't"", 'RB'), ('apply', 'VB'), ('Weakness', 'NN'), ('and', 'CC'), ('Resistance', 'NN'), ('for', 'IN'), ('Benched', 'VBN'), ('Pokémon', 'NNP'), ('.', '.'), (')', '-RRB-')]","[('this', 'DT'), ('attack', 'NN'), ('do', 'VBZ'), ('100', 'CD'), ('damage', 'NN'), ('to', 'IN'), ('1', 'CD'), ('of', 'IN'), ('your', 'PRP$'), ('opponent', 'NN'), (""'s"", 'POS'), ('pokémon', 'NNP'), ('.', '.'), ('(', '-LRB-'), ('do', 'VB'), (""n't"", 'RB'), ('apply', 'VB'), ('weakness', 'NN'), ('and', 'CC'), ('resistance', 'NN'), ('for', 'IN'), ('benched', 'VBN'), ('pokémon', 'NNP'), ('.', '.'), (')', '-RRB-')]",3.0,"['pokémon', 'deck', 'card', 'shuffle', 'basic', 'search', 'hand', 'evolve', 'player', 'turn']",do_damage100,attack,player,opponent Pokémon,None
7,Keito Arai,Professor's Research,Trainer,NaN,NaN,NaN,Discard your hand and draw 7 card,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,"[('Discard', 'VB'), ('your', 'PRP$'), ('hand', 'NN'), ('and', 'CC'), ('draw', 'VB'), ('7', 'CD'), ('cards', 'NNS'), ('.', '.')]","[('discard', 'VB'), ('your', 'PRP$'), ('hand', 'NN'), ('and', 'CC'), ('draw', 'VB'), ('7', 'CD'), ('card', 'NNS'), ('.', '.')]",NaN,"['pokémon', 'deck', 'card', 'shuffle', 'basic', 'search', 'hand', 'evolve', 'player', 'turn']",discard+draw,hand,player,card(7),None
8,Keito Arai,Iono,Trainer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Keito Arai,Arven,Trainer,NaN,NaN,NaN,Search your deck for an Item card and a Pokémon Tool card reveal them and put them into your hand Then shuffle your deck,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,"[('Search', 'VB'), ('your', 'PRP$'), ('deck', 'NN'), ('for', 'IN'), ('an', 'DT'), ('Item', 'NN'), ('card', 'NN'), ('and', 'CC'), ('a', 'DT'), ('Pokémon', 'NNP'), ('Tool', 'NN'), ('card', 'NN'), (',', ','), ('reveal', 'VB'), ('them', 'PRP'), (',', ','), ('and', 'CC'), ('put', 'VB'), ('them', 'PRP'), ('into', 'IN'), ('your', 'PRP$'), ('hand', 'NN'), ('.', '.'), ('Then', 'RB'), (',', ','), ('shuffle', 'VB'), ('your', 'PRP$'), ('deck', 'NN'), ('.', '.')]","[('search', 'VB'), ('your', 'PRP$'), ('deck', 'NN'), ('for', 'IN'), ('an', 'DT'), ('item', 'NN'), ('card', 'NN'), ('and', 'CC'), ('a', 'DT'), ('pokémon', 'NNP'), ('tool', 'NN'), ('card', 'NN'), (',', ','), ('reveal', 'VB'), ('them', 'PRP'), (',', ','), ('and', 'CC'), ('put', 'VB'), ('them', 'PRP'), ('into', 'IN'), ('your', 'PRP$'), ('hand', 'NN'), ('.', '.'), ('then', 'RB'), (',', ','), ('shuffle', 'VB'), ('your', 'PRP$'), ('deck', 'NN'), ('.', '.')]",NaN,"['pokémon', 'deck', 'card', 'shuffle', 'basic', 'search', 'hand', 'evolve', 'player', 'turn']",search,deck,player,